# Звіт

## 1. Задача і чому потрібне донавчання
Ціль: класифікувати англійський заголовок новини як сатиричний (1) чи справжній (0). Таким класифікатором можуть користуватися агрегатори новин і соцмережі: він позначає сатиру, щоб її не сприймали за реальну новину й не поширювали як дезінформацію. На вхід подається один заголовок, на виході мітка з імовірністю. Сарказм базується на стилістичних ознаках: абсурдні деталі, пародійна серйозність, перебільшення. Описати їх промптом тяжко. Для класифікатора, що обробляє потік заголовків, важливі швидкість і стабільність, і мала модель-енкодер дає їх краще за генеративну LLM. RAG тут не допомагає, бо відповідь залежить від стилю тексту, а не від зовнішніх знань.

## 2. Модель і дані
roberta-base (125 млн параметрів) з новою головою на 2 класи. Задача вузька, генеративна модель не потрібна. Чому не звичайна BERT: бо RoBERTa тренувалася на більшому корпусі та зокрема на новинах.

Дані: [News Headlines Dataset For Sarcasm Detection](https://github.com/rishabhmisra/News-Headlines-Dataset-For-Sarcasm-Detection), файл Sarcasm_Headlines_Dataset.json. Ліцензію в репозиторії не вказано, автор просить цитувати статтю «Sarcasm Detection using News Headlines Dataset» (AI Open, 2023).

Формат: JSON Lines з полями is_sarcastic, headline, article_link. 28 619 записів , після дедуплікації 28 503, з них 13 552 сатиричних (47,5%), класи майже збалансовані. Поділ 80/10/10 зі стратифікацією: 22 802 / 2 850 / 2 851 прикладів, частка сарказму в кожному наборі 47,5%. Обмеження: усі сатиричні заголовки з The Onion, а всі справжні з HuffPost, тому модель може частково вчитися стилю двох редакцій, а не сарказму загалом.

## 3. Метрика й результат до донавчання
- **Accuracy:** класи майже збалансовані, тому вона інформативна.
- **F1 для класу «сарказм»:** модель, яка ніколи не знаходить сарказм, отримує F1 = 0 навіть за accuracy понад 50%.

### Результат на тестовому наборі

| Модель | Accuracy | F1 | Loss |
|---|---|---|---|
| Завжди найчастіший клас (0) | 0,524 | 0 | — |
| roberta-base з ненавченою головою | 0,524 | 0 | 0,694 |

Ненавчена модель повністю збігається з базовим рівнем. Loss ≈ ln 2, як при випадковому вгадуванні. На всіх прикладах вона дає майже однакову ймовірність сарказму, близько 44%, тобто завжди відповідає «не сарказм»: нова голова ініціалізована випадково і ще не вміє використовувати представлення кістяка.

### Приклади

| Заголовок | Мітка | Сарказм |
|---|---|---|
| londoners 2 percent less polite about terrorism following bombings | сарказм | 44,0% |
| plan to make snacks last through opening credits fails | сарказм | 44,1% |
| nasa celebrates 60th anniversary of launching first moon to orbit earth | сарказм | 43,9% |
| 5 types of annoying people to avoid at all costs | не сарказм | 44,2% |


## 4. Метод донавчання й гіперпараметри
Було взято повне донавчання з Trainer: тренуються всі 124 647 170 параметрів (кістяк і нова голова). Модель на 125 млн параметрів вміщується в T4, тому LoRA не потрібна.

### Спроба 1

learning_rate=2e-5 (малий крок, щоб не зруйнувати знання з попереднього тренування), 3 епохи, батч 16, fp16, оцінка на валідації після кожної епохи й завантаження чекпоінта з найменшим validation loss. Тривалість 6,6 хв, пікова памʼять GPU 3,02 GB.

| Епоха | Training loss | Validation loss | Accuracy (вал.) | F1 (вал.) |
|---|---|---|---|---|
| 1 | 0,245 | 0,208 | 0,922 | 0,916 |
| 2 | 0,150 | 0,285 | 0,924 | 0,915 |
| 3 | 0,104 | 0,341 | 0,931 | 0,925 |

Модель перенавчається з другої епохт.

### Спроба 2

2 епохи, щоб learning rate спадав до нуля швидше, і weight_decay=0.01 як легка регуляризація. Тривалість 3,9 хв, пікова памʼять GPU 3,51 GB. Це значення трохи завищене: навіть після очищення в GPU, схоже, лишилася частина моделі першої спроби.

| Епоха | Training loss | Validation loss | Accuracy (вал.) | F1 (вал.) |
|---|---|---|---|---|
| 1 | 0,239 | 0,241 | 0,913 | 0,903 |
| 2 | 0,126 | 0,273 | 0,925 | 0,918 |

Модель так само перенавчається з другої епохи.

**Обрано** чекпоінт 1-ї епохи спроби 1 (validation loss 0,208, accuracy 0,922, F1 0,916). На 22,8 тис. заголовків модель засвоює все корисне за одну епоху, подальше тренування змушує модель перенавчатися.

## 5. Результат після донавчання
### Тестовий набір

| Модель | Accuracy | F1 | Loss |
|---|---|---|---|
| До донавчання | 0,524 | 0 | 0,694 |
| Після донавчання | 0,932 | 0,926 | 0,178 |

Результати на тесті близькі до валідаційних (0,922 / 0,916), тобто вибір чекпоінта за валідацією не підігнав модель під конкретний набір.

### Матриця помилок

| | Передбачено «не сарказм» | Передбачено «сарказм» |
|---|---|---|
| Справді не сарказм | 1439 | 56 |
| Справді сарказм | 139 | 1217 |

Precision для сарказму 95,6%, recall 89,7%: модель рідко помилково називає новину сатирою, але пропускає кожен десятий сатиричний заголовок.

Для прикладів з меж датасету: правильно класифіковано 6 з 8, до донавчання лише 3. Сарказм модель впевнено розпізнала з імовірністю 98,8% і 96,4%, а справжні новини оцінила в 0,4–0,6% сарказму.


## 6. Аналіз помилок
Помилок всього 195 з 2 851 (6,8%). Хибно позитивні (справжня новина -> сарказм) бувають трьох видів:

- Перший: реальні новини, які звучать абсурдно: «new mayor drives around in giant snail car» (99,7%), «passport robot tells man of asian descent his eyes are too closed» (98,4%). Без знання, що це сталося насправді, їх неможливо відрізнити від сатири.
- Другий: короткі сухі кримінальні заголовки («9-month old shot by father cleaning illegal gun», «son possibly made withdrawal with dead mom»), схожі на чорний гумор The Onion.
- Третій: політичні заголовки про Обаму, Клінтон і Трампа, часті мішені The Onion, тож модель повʼязала тему з міткою.
Хибно негативні (сарказм -> новина) теж поділяються на групи:
- Перша: сатира, що пародіює нейтральні формати новин без явних маркерів іронії: «inside: what the stars were wearing at terrible movie's gala premiere», «special pull-out section: ...», «disney's 'toy tales' hits theaters friday».
- Друга: жарти, для яких потрібні знання про світ: «stan lee, creator of beloved marvel character stan lee, dead at 95» (натяк на його камео у фільмах), «billy joel has billy joel's disease».

### Перевірка на власних заголовках

| Заголовок | Задумано як | Сарказм |
|---|---|---|
| city council approves budget for new public library | новина | 84,2% |
| scientists discover new species of frog in peruvian rainforest | новина | 46,7% |
| local man / area dad / nation's dogs ... | сатира | 99,7–99,8% |
| study finds most office meetings could have been an email | пограничний | 99,6% |
| 10 easy dinner recipes for busy weeknights | новина | 0,3% |

Перевірка підтвердила обмеження даних. Місцеві новини й «науковці» - типові теми The Onion, тож нейтральні заголовки на ці теми модель вважає сарказмом. Формули The Onion («local man», «area dad», «nation's», «study finds») дають майже 100% сарказму, а лістикл у стилі HuffPost - 0,3%. Отже, модель значною мірою розпізнає стиль і типові теми двох видань, а не сарказм як такий.


## 7. Висновок
Донавчання roberta-base з новою головою підняло accuracy з рівня вгадування (0,524) до 0,932 і F1 з 0 до 0,926. Для набору даних достатньо однієї епохи, далі модель перенавчається, а зміна гіперпараметрів не покращила якість. Головне обмеження не в моделі, а в даних: кожен клас походить з одного видання, тому класифікатор вчиться стилю видання й помиляється на нейтральних заголовках з «онівських» тем та на сатирі, що імітує звичайні новини. Для практичного застосування важливіше розширити дані сатирою й новинами з різних джерел, ніж підбирати гіперпараметри. Поріг класифікації варто налаштувати під задачу: для маркування сатири в стрічці краще вищий recall, для автоматичних дій — вищий precision.

Модель: [NexLiR/roberta-base-sarcasm-headlines](https://huggingface.co/NexLiR/roberta-base-sarcasm-headlines).

# Лабораторна 5. Донавчання. Виявлення сарказму в заголовках новин

Донавчаємо `roberta-base` з новою головою-класифікатором визначати, чи є заголовок новини сатиричним чи справжнім.

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані

[News Headlines Dataset For Sarcasm Detection](https://github.com/rishabhmisra/News-Headlines-Dataset-For-Sarcasm-Detection) - заголовки з сатиричного сайту The Onion 1 і справжні новини з HuffPost 0.
Кожен запис має поля is_sarcastic, headline і article_link.

In [3]:
DATA_URL = (
    "https://raw.githubusercontent.com/rishabhmisra/News-Headlines-Dataset-For-Sarcasm-Detection/"
    "master/Sarcasm_Headlines_Dataset.json"
)
raw_dataset = load_dataset("json", data_files=DATA_URL, split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(8)):
    print(f"{ex['is_sarcastic']}  {ex['headline']}")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['is_sarcastic', 'headline', 'article_link'],
    num_rows: 28619
})
0  kevin hart drops an f-bomb in awkward nfl network interview
0  new photos of kit harington give 'got' fans hope
1  maya angelou thought she'd be invited to more white house stuff
1  astronomers admit they made neptune up
1  next episode of 'girls' to feature lena dunham shitting herself during gyno exam while eating a burrito
0  best hotels for large families
0  the two non-interventionists
0  why there are tiny holes at the bottom of windows on planes


### Дедуплікація

Залишаємо по одному прикладу для кожного заголовка, щоб однакові заголовки не потрапили одночасно в тренувальний і тестовий набори.

In [4]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["headline"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

labels = np.array(dedup_dataset["is_sarcastic"])
print(f"Сатиричних: {labels.sum()} ({labels.mean():.1%}), справжніх: {len(labels) - labels.sum()}")

До дедуплікації: 28619, після: 28503
Сатиричних: 13552 (47.5%), справжніх: 14951


### Тренувальний, валідаційний і тестовий набори

Ділимо 80/10/10 зі стратифікацією, щоб частка сарказму в усіх наборах була однаковою.
Тестовий набір використовуємо лише для підсумкової оцінки до й після донавчання.

In [5]:
dataset = (
    dedup_dataset.remove_columns("article_link")
    .rename_column("is_sarcastic", "label")
    .class_encode_column("label")
)

split = dataset.train_test_split(test_size=0.2, stratify_by_column="label", seed=42)
val_test = split["test"].train_test_split(test_size=0.5, stratify_by_column="label", seed=42)
train_dataset, val_dataset, test_dataset = split["train"], val_test["train"], val_test["test"]

assert not set(train_dataset["headline"]) & set(test_dataset["headline"])

for name, ds in [("Тренувальний", train_dataset), ("Валідаційний", val_dataset), ("Тестовий", test_dataset)]:
    print(f"{name}: {len(ds)}, частка сарказму {np.mean(ds['label']):.1%}")

Stringifying the column:   0%|          | 0/28503 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/28503 [00:00<?, ? examples/s]

Тренувальний: 22802, частка сарказму 47.5%
Валідаційний: 2850, частка сарказму 47.5%
Тестовий: 2851, частка сарказму 47.6%


## 2. Модель до донавчання

### Токенізація і модель з новою головою

Як у демо 1: токенізуємо заголовки, а `AutoModelForSequenceClassification` додає до кістяка RoBERTa нову голову на 2 класи з випадковими вагами.

In [6]:
checkpoint = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(batch):
    return tokenizer(batch["headline"], truncation=True)


train_tok = train_dataset.map(tokenize_function, batched=True)
val_tok = val_dataset.map(tokenize_function, batched=True)
test_tok = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
print(model.classifier)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/22802 [00:00<?, ? examples/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=2, bias=True)
)


### Метрики

- **accuracy** — частка правильно класифікованих заголовків; класи майже збалансовані (47,5% сарказму), тому вона інформативна;
- **F1 для класу «сарказм»** — враховує і precision, і recall саме для сатиричних заголовків. Модель, яка ніколи не знаходить сарказм, отримує F1 = 0 навіть за високої accuracy.

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


test_labels = np.array(test_dataset["label"])
majority_class = np.bincount(train_dataset["label"]).argmax()
majority_predictions = np.full_like(test_labels, majority_class)
print(f"Найчастіший клас у тренувальному наборі: {majority_class}")
print(f"Базовий рівень на тесті: accuracy {accuracy_score(test_labels, majority_predictions):.3f}, "
      f"F1 {f1_score(test_labels, majority_predictions):.3f}")

Найчастіший клас у тренувальному наборі: 0
Базовий рівень на тесті: accuracy 0.524, F1 0.000


In [8]:
baseline_args = TrainingArguments(
    output_dir="roberta-sarcasm-baseline",
    per_device_eval_batch_size=64,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
baseline_trainer = Trainer(
    model=model,
    args=baseline_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
baseline_trainer.evaluate(test_tok)

Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.694187,0,0.524377,0.000000


{'eval_loss': 0.6941867470741272,
 'eval_accuracy': 0.5243774114345844,
 'eval_f1': 0.0}

### Приклади відповідей моделі до донавчання

In [9]:
def predict(model, headline):
    model.eval()
    inputs = tokenizer(headline, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"сарказм: {probs[1]:.1%}"


examples = test_dataset.select(range(8))
for ex in examples:
    print(f"[{ex['label']}] {ex['headline']}\n→ {predict(model, ex['headline'])}\n")

[1] londoners 2 percent less polite about terrorism following bombings
→ сарказм: 44.0%

[0] before the end of the year...
→ сарказм: 43.9%

[1] plan to make snacks last through opening credits fails
→ сарказм: 44.1%

[1] indonesian mother sews halloween costumes for 60,000 children
→ сарказм: 43.9%

[1] cocktail party gets as wild as it's going to get
→ сарказм: 43.8%

[1] nasa celebrates 60th anniversary of launching first moon to orbit earth
→ сарказм: 43.9%

[0] 5 types of annoying people to avoid at all costs
→ сарказм: 44.2%

[0] timothée chalamet says contract blocked him from criticizing woody allen, supporting dylan farrow. it didn't.
→ сарказм: 43.6%



## 3. Донавчання

Повне донавчання з Trainer: оновлюються всі ваги, і кістяк, і нова голова.
Гіперпараметри:
- learning_rate=2e-5 - малий крок, щоб не зіпсувати знання, отримані під час попереднього тренування;
- num_train_epochs=3 - нормальна кількість епох, для повного донавчання;
- per_device_train_batch_size=16 - вміщується в памʼять T4 і дає достатньо кроків оновлення на 22,8 тис. прикладах.

Після кожної епохи зберігаємо чекпоінт і наприкінці завантажуємо той, що має найменший validation loss (load_best_model_at_end=True)

In [10]:
import time

training_args = TrainingArguments(
    output_dir="roberta-sarcasm",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [11]:
print(f"Параметрів, що тренуються: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer.train()
print(f"Тривалість тренування: {(time.time() - start) / 60:.1f} хв")
print(f"Пікова памʼять GPU: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

trainer.evaluate()

Параметрів, що тренуються: 124,647,170


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.244604,0.208101,0.922105,0.915909
2,0.150231,0.285482,0.923509,0.914710
3,0.104184,0.340789,0.931228,0.925248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Тривалість тренування: 6.6 хв
Пікова памʼять GPU: 3.02 GB


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.104184,0.208101,3,0.922105,0.915909


{'eval_loss': 0.20810067653656006,
 'eval_accuracy': 0.9221052631578948,
 'eval_f1': 0.9159090909090909}

### Перенавчання і друга спроба

Validation loss найменший після 1-ї епохи (0,204) і далі зростає до 0,343, хоча training loss падає: модель перенавчається і стає надмірно впевненою на помилках.
Чекпоінт 1-ї епохи збережено, коли learning rate ще не спав до нуля, тому пробуємо коротший розклад і легку регуляризацію:
- num_train_epochs=2 - третя епоха зайва;
- weight_decay=0.01 — штраф на великі ваги (за замовчуванням вимкнений).

Решта параметрів без змін.

In [12]:
import gc

del trainer, model
gc.collect()
torch.cuda.empty_cache()

set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="roberta-sarcasm-v2",
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer.train()
print(f"Тривалість тренування: {(time.time() - start) / 60:.1f} хв")
print(f"Пікова памʼять GPU: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.239226,0.241294,0.912632,0.903000
2,0.126379,0.272859,0.924912,0.917502


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Тривалість тренування: 3.9 хв
Пікова памʼять GPU: 3.51 GB


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.126379,0.241294,2,0.912632,0.903000


{'eval_loss': 0.24129395186901093,
 'eval_accuracy': 0.9126315789473685,
 'eval_f1': 0.9029996104402026}

## 4. Порівняння до і після донавчання

In [14]:
import os

import pandas as pd
from sklearn.metrics import confusion_matrix

print(os.listdir("roberta-sarcasm"))

del trainer, model, baseline_trainer
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained("roberta-sarcasm/checkpoint-1426").to(device)

eval_trainer = Trainer(
    model=model,
    args=baseline_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
test_output = eval_trainer.predict(test_tok)
test_output.metrics

['checkpoint-1426']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'test_loss': 0.17820408940315247,
 'test_model_preparation_time': 0.0054,
 'test_accuracy': 0.9316029463346195,
 'test_f1': 0.9258273107645493,
 'test_runtime': 1.3712,
 'test_samples_per_second': 2079.213,
 'test_steps_per_second': 32.818}

In [15]:
pd.DataFrame({
    "Найчастіший клас": {
        "accuracy": accuracy_score(test_labels, majority_predictions),
        "f1": f1_score(test_labels, majority_predictions),
    },
    "Після донавчання": {
        "accuracy": test_output.metrics["test_accuracy"],
        "f1": test_output.metrics["test_f1"],
    },
}).round(3)

,Найчастіший клас,Після донавчання
accuracy,0.524,0.932
f1,0.000,0.926


### Ті самі приклади, що й до донавчання

In [16]:
for ex in examples:
    print(f"[{ex['label']}] {ex['headline']}\n→ {predict(model, ex['headline'])}\n")

[1] londoners 2 percent less polite about terrorism following bombings
→ сарказм: 96.4%

[0] before the end of the year...
→ сарказм: 0.5%

[1] plan to make snacks last through opening credits fails
→ сарказм: 98.8%

[1] indonesian mother sews halloween costumes for 60,000 children
→ сарказм: 22.2%

[1] cocktail party gets as wild as it's going to get
→ сарказм: 10.4%

[1] nasa celebrates 60th anniversary of launching first moon to orbit earth
→ сарказм: 65.7%

[0] 5 types of annoying people to avoid at all costs
→ сарказм: 0.4%

[0] timothée chalamet says contract blocked him from criticizing woody allen, supporting dylan farrow. it didn't.
→ сарказм: 0.6%



### Аналіз помилок

Матриця помилок (рядки — справжня мітка, стовпці — передбачена) і найупевненіші помилки кожного типу:
- хибно позитивні — справжня новина, яку модель вважає сарказмом;
- хибно негативні — сарказм, який модель не розпізнала.

In [17]:
probs = torch.tensor(test_output.predictions).softmax(dim=-1)[:, 1].numpy()
preds = np.argmax(test_output.predictions, axis=-1)
print(confusion_matrix(test_labels, preds))

headlines = test_dataset["headline"]
error_ids = np.where(preds != test_labels)[0]
false_positives = sorted((i for i in error_ids if test_labels[i] == 0), key=lambda i: -probs[i])
false_negatives = sorted((i for i in error_ids if test_labels[i] == 1), key=lambda i: probs[i])

print(f"\nХибно позитивні: {len(false_positives)}")
for i in false_positives[:10]:
    print(f"  сарказм {probs[i]:.1%}  {headlines[i]}")

print(f"\nХибно негативні: {len(false_negatives)}")
for i in false_negatives[:10]:
    print(f"  сарказм {probs[i]:.1%}  {headlines[i]}")

[[1439   56]
 [ 139 1217]]

Хибно позитивні: 56
  сарказм 99.7%  new mayor drives around in giant snail car
  сарказм 98.9%  wall street journal editor directs reporters to get really mealy-mouthed covering trump
  сарказм 98.5%  guard dog in training notices very suspicious-looking dog in mirror
  сарказм 98.4%  passport robot tells man of asian descent his eyes are too closed
  сарказм 98.4%  obama begins sales pitch on trade to wary u.s. public
  сарказм 97.6%  son possibly made withdrawal with dead mom
  сарказм 96.9%  angry voters not soothed by clinton's policy prescriptions
  сарказм 96.6%  9-month old shot by father cleaning illegal gun
  сарказм 96.6%  medical examiner releases amonderez green autopsy
  сарказм 96.1%  senate democrats already willing to work with trump administration

Хибно негативні: 139
  сарказм 0.4%  hootie and the blowfish: breaking down racial barriers between black, white pussies
  сарказм 0.6%  inside: what the stars were wearing at terrible movie's ga

### Перевірка на власних заголовках

Усі сатиричні заголовки датасету — з The Onion, а справжні — з HuffPost.
Перевіряємо на власних заголовках, чи модель розпізнає сарказм, чи лише стиль двох видань:
перші три написані як серйозні новини, наступні три — як сатира, останні два — пограничні випадки.

In [18]:
my_headlines = [
    "central bank raises interest rates by a quarter point amid inflation concerns",
    "scientists discover new species of frog in peruvian rainforest",
    "city council approves budget for new public library",
    "local man confident he can fix printer by staring at it harder",
    "area dad announces he will be taking the scenic route again",
    "nation's dogs vow to keep barking at mailman until he finally explains himself",
    "study finds most office meetings could have been an email",
    "10 easy dinner recipes for busy weeknights",
]
for headline in my_headlines:
    print(f"{headline}\n→ {predict(model, headline)}\n")

central bank raises interest rates by a quarter point amid inflation concerns
→ сарказм: 3.5%

scientists discover new species of frog in peruvian rainforest
→ сарказм: 46.7%

city council approves budget for new public library
→ сарказм: 84.2%

local man confident he can fix printer by staring at it harder
→ сарказм: 99.8%

area dad announces he will be taking the scenic route again
→ сарказм: 99.8%

nation's dogs vow to keep barking at mailman until he finally explains himself
→ сарказм: 99.7%

study finds most office meetings could have been an email
→ сарказм: 99.6%

10 easy dinner recipes for busy weeknights
→ сарказм: 0.3%



In [19]:
from huggingface_hub import notebook_login

notebook_login()

In [20]:
repo_id = "NexLiR/roberta-base-sarcasm-headlines"
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hdb6gac/model.safetensors:   0%|          |  623kB /  499MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/NexLiR/roberta-base-sarcasm-headlines/commit/5e68b4ce3e3b327a83870f579f9610a4d59a707c', commit_message='Upload tokenizer', commit_description='', oid='5e68b4ce3e3b327a83870f579f9610a4d59a707c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/NexLiR/roberta-base-sarcasm-headlines', endpoint='https://huggingface.co', repo_type='model', repo_id='NexLiR/roberta-base-sarcasm-headlines'), pr_revision=None, pr_num=None)